# AIショッピングアシスタントの評価を構築する

このビルドアロングでは、AIエージェントの**評価（eval）を構築**し、それを使って問題を体系的に見つけて直す方法を学びます。

`boutique` は、商品の価格を調べて計算をするシンプルなショッピングアシスタントです。おおむね動きますが、完璧ではありません。どこで、なぜ壊れるのかを正確に突き止めて、直すのが皆さんの役割です。

---

**Partner Basecamp 参加者向けに作成しています。** 研修教材としての複製・再配布はできません。ただし、ここで学ぶパターンをお客様の案件で活用していただくのは自由です。

In [ ]:
# このカーネルに依存パッケージをインストールします。再実行しても安全で、制限の厳しい（PEP 668）Pythonでも動作します。
import importlib.util, os, subprocess, sys

# ── 環境ガード（Environment guard）─────────────────────────────────────
# 下の installer と同じセルに置いているので、スキップできません。
# 素のシステムPythonにパッケージがインストールされることは一切ありません（以前の最悪のケースは、
# IT管理のマシンに --break-system-packages を使うことでした）。ここで止まった場合は、
# SETUP.md の「Why the notebook just stopped」を参照してください。
def _in_isolated_env():
    """実行中のカーネルが venv／virtualenv／conda 環境、またはColabの場合にTrueを返します。
    インタープリター自体（sys.*）から判断します。起動元のシェルから継承した有効化用の環境変数は、
    sys.executable が実際にその環境の内側にある場合にのみ信頼します。有効化済みのターミナルから
    起動したシステムPythonのカーネルも VIRTUAL_ENV を継承しますが、
    通過させてはいけません。"""
    if "google.colab" in sys.modules:
        return True  # Colabは専用の使い捨てランタイムで動くため分離されている
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405 の venv（python -m venv）。ほとんどの conda 環境も該当
    if hasattr(sys, "real_prefix"):
        return True  # 旧式の virtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # シェルから継承しただけの情報（hearsay）で、カーネルは別の場所にある
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # conda の共有 `base` は分離環境に含めない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードをバイパスしました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "このPythonに意図的にインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムのPythonを検出しました。何もインストールせずに停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変わります。会社管理の\n"
        "ノートパソコンでは、ITへの問い合わせが必要になることもあります。\n"
        "\n"
        "対処法（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから次を実行します：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Code で、右上のカーネル名をクリック → Select Another Kernel →\n"
        "     Python Environments → 末尾が .venv のものを選択 → このセルをもう一度実行します。\n"
        "\n"
        "conda を使っている場合は、`conda activate <env>`（`base` 以外）を実行してから、そのカーネルを選びます。\n"
        "自己管理のマシンを使うファシリテーターは、BASECAMP_ALLOW_SYSTEM_PYTHON=1 でバイパスできます。"
    )
    _shown = False
    try:  # 下のAPIキー確認と同じバナー表現（緑のボックス、赤のバリエーション）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードがこのセルを停止しました。上のメッセージを確認してください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：分離されたインタープリターを検出しました。インストールしても安全です")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: (import_name, pip_spec) のリスト。足りないものだけを、実行中の
    インタープリターにインストールします。通常のインストール、ユーザー領域、
    PEP 668 のオーバーライド（まずユーザー領域、システム全体は最後の手段）の順に試します。
    どの試行も出力は捕捉するだけでストリーム表示しないので、制限の厳しい
    Python（HomebrewやDebian、PEP 668）でも、フォールバックが成功したときに
    'externally-managed-environment' の長いエラーが出ません。すべての方法が
    失敗したときだけ、生のトレースバックではなく venv での対処法とともに理由を表示します。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + "（初回のみ、しばらくお待ちください）…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "（pipからの出力なし）"
    raise SystemExit(
        "\n  インストールできませんでした：" + ", ".join(missing) + "\n"
        "  このPythonは制限されている（PEP 668）か、オフラインです。最も手早い対処は venv です：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windows は SETUP.md を参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code（右上のカーネルピッカー）で .venv のインタープリターを選び、Run All を実行してください。\n"
        "  社内プロキシやPyPIのブロックが原因ですか？ リポジトリのルートにある SETUP.md を参照してください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ Dependencies ready")

### セットアップ：Claudeに接続する

次のセルを最初に実行してください。セットアップセルは、初回の実行時に **`.env` ファイル**を作成します（gitignore 対象のため、キーがコミットされることはありません）。ファイルを開き、`ANTHROPIC_API_KEY=` の後ろにキーを貼り付けて保存し、もう一度実行します。カーネルを再起動しても残るので、貼り付けは1回で済みます。*（`.env` がまだない場合は、代わりに非表示の入力ボックスが表示されます。）* 緑色の**「✓ API key verified」**バナーが表示されれば完了です。赤いバナーが出た場合は、表示された指示に従って、もう一度セルを実行してください。

In [ ]:
import os

def _status(ok, msg):
    """ノートブックでは緑／赤のバナー、スクリプト実行時はプレーンテキストを表示します。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックのカーネルではありません。プレーンテキストのバナーを使います")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeに接続する：Anthropic API または Amazon Bedrock ──
# どちらの認証情報（API key / Bedrock token）でも動作します。セルが、どちらを持っているかを判別します。
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  plus  AWS_REGION=us-east-1
# 使う方を、このセルが作成する .env ファイル（gitignore 対象のため、コミットされません）に入れてください。
# シェルで export しても構いません。シェルの値が .env ファイルの値より優先されます。
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """作業ディレクトリから上に向かって、最も近い既存の .env を探します（ルートの .env 1つで全演習をまかなえます）。
    まだなければリポジトリのルートを指し、ノートブックだけを単体で開いている場合は
    このフォルダを指します。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# 小さな .env パーサー（python-dotenv は不要）。実行のたびに読み直すので、キーを貼り付けて
# 再実行すれば反映されます。環境（シェル / Claude Code / CI）にある実際の値が
# 優先され、placeholder が残ることはありません。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """警告の黄色いバナーを表示して停止します。数セル先ではなく、ここでセットアップを失敗させるためです。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報がありません。上のメッセージを確認してください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続行するには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic APIを使う場合は、次を設定します：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrockを使う場合は、両方を設定します：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効化しているリージョン\n"
        "\n"
        "ファイルを保存し、このセルの ▶ をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrockにはリージョンが必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrockのモデルを有効化しているリージョンを追加してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存し、このセルの ▶ をもう一度クリックしてください。"
    )


def _model(name):
    """BedrockのモデルIDには `anthropic.` のプレフィックスが付きます。Anthropic APIでは付けないIDをそのまま使います。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# Named models：この演習で使うモデルを、利用中のプロバイダーに合わせて解決します。
MODEL = _model("claude-sonnet-5")        # この演習の主力（workhorse）
FAST_MODEL = _model("claude-haiku-4-5")  # 安くて速い（接続確認、judge 用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなモデル（larger model）を試したいとき


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# 接続確認：credential と、このモデルを利用できることの両方を検証します。
# Bedrock では、モデルがアカウント／リージョンで有効化されていないと、有効なキーでも404になることがあります。
# そのため、認証情報の形式だけでなく、実際のモデルIDにpingを送って確認します。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルを利用できません。上のメッセージを確認してください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "そのBedrockキーは拒否されました。AWS_BEARER_TOKEN_BEDROCK を確認し、"
                       "Bedrockの呼び出し権限があることも確認してから、このセルをもう一度実行してください。")
    else:
        _status(False, "そのキーは拒否されました。このセルをもう一度実行し、キー全体を貼り付けてください"
                       "（sk-ant- で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルをもう一度実行してください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。"
                   "接続状況を確認して、このセルをもう一度実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 後続のセルや !python からも参照できるようにする
        _status(True, "API key verified（APIキーを確認しました）。Claudeに接続できています。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 本番用クライアント。タイムアウトを長くしているのは、ストリーミングなしの呼び出しで max_tokens>21333 にするために必要なためです。
client = _make_client(timeout=900.0)


## なぜ評価が必要か

Claudeで開発しているとき、「何回か試して動けばOK」という進め方は最善ではありません。感覚ももちろん大切ですが、評価があれば次のものが手に入ります。

- **ベースライン。** エージェントは今どのくらい良いのか。どの種類のクエリで、どうなのか。
- **フィードバックループ。** プロンプト、ツール設計、モデルのいずれかを変えて再実行し、本当に改善したか、退行（リグレッション）がないかを確認できます。
- **確信。** お客様に出す前に、感覚ではなく数字で示せます。

今日のビルドアロングの流れ：

1. **エージェントを知り**、何が動いて何が動かないかに気づき始める
2. エージェントの機能とエッジケースをカバーする**評価タスクを定義する**
3. **評価を実行し**、結果を体系的に調べる
4. 評価結果を使って**エージェントを改善し**、再実行して確かめる
5. 単純な文字列一致では確認できないクエリのために、**LLM-as-Judge 採点者を追加する**

---

## パート1：エージェントを知る

`boutique` は、ツールを2つ持つシンプルな単一ターンのエージェントです。

| ツール | 役割 |
|------|-------------|
| `get_product(product)` | カタログから商品の価格を返す |
| `calculate(op, input1, input2)` | 基本的な計算 |

エージェントループはシンプルです。

```
User query → Claude decides what to do → Tool call (if needed) → Tool result → ... → Final answer
```

下のコードにざっと目を通してください。あとで見返したくなる箇所があるかもしれません。
- **ツール仕様**（Claudeが各ツールについて見て知っている情報）
- **システムプロンプト**
- **ツールの実装**（ツールが呼ばれたときに実際に起きること）

In [ ]:
import math
from anthropic import Anthropic
from anthropic.types import ToolUseBlock, TextBlock

# ── 設定（Config）────────────────────────────────────────────────────

MODEL = FAST_MODEL  # Haiku。セットアップで、利用中のプロバイダーに合わせて解決済み（エイリアスなので、廃止されても404にならない）
SYSTEM_PROMPT = "You are a helpful assistant."

# ハーネス用のクライアント：SDKのデフォルト（長いタイムアウト、標準のリトライ）。上の確認用
# クライアントは30秒のタイムアウトと1回のリトライという短い設定で、1トークンのpingには適していますが、
# 評価（eval）を最後まで実行するには神経質すぎます。作り直します。key はすでに環境にあります。
client = Anthropic()

# ── ツールの実装（Tool implementations）──────────────────────────────

def get_product(product: str):
    catalog = {
        "jeans": 49.99,
        "shirt": 29.99,
        "dress": 59.99,
        "jacket": 89.99,
        "sneakers": 74.99,
        "hat": 19.99,
        "socks": 9.99,
        "hoodie": 44.99,
        "shorts": 34.99,
        "t-shirt": 24.99,
        "sweater": 54.99,
        "belt": 24.99,
    }
    return catalog[product]


def calculate(op: str, input1: float, input2: float):
    if op == "+": return input1 + input2
    elif op == "-": return input1 - input2
    elif op == "*": return input1 * input2
    elif op == "/": return input1 / input2
    elif op == "**": return input1 ** input2

TOOL_REGISTRY = {
    "get_product": get_product,
    "calculate": calculate,
}

# ── ツール仕様（Claudeに送る）──────────────────────────────────────────────

GET_PRODUCT_SPEC = {
    "name": "get_product",
    "description": "get_product",
    "input_schema": {
        "type": "object",
        "properties": {
            "product": {
                "type": "string",
                "description": "product",
            },
        },
        "required": ["product"],
    },
}

CALCULATE_SPEC = {
    "name": "calculate",
    "description": "calculator",
    "input_schema": {
        "type": "object",
        "properties": {
            "op": {
                "type": "string",
                "description": "operator",
            },
            "input1": {
                "type": "number",
                "description": "input1",
            },
            "input2": {
                "type": "number",
                "description": "input2",
            },
        },
        "required": ["op", "input1", "input2"],
    },
}

ALL_TOOL_SPECS = [GET_PRODUCT_SPEC, CALCULATE_SPEC]

# ── エージェント（Agent）─────────────────────────────────────────────

def call_claude(messages, tools, model=None):
    return client.messages.create(
        model=model or MODEL,
        system=SYSTEM_PROMPT,
        max_tokens = 1024,
        tools=tools,
        messages=messages,
    )


def execute_tool(name, inputs):
    try:
        return str(TOOL_REGISTRY[name](**inputs))
    except Exception as e:
        return f"Error: {e}"


def run_agent(prompt, eval_mode=False, model=None):
    messages = [{"role": "user", "content": prompt}]
    total_input_tokens = 0
    total_output_tokens = 0

    while True:
        response = call_claude(messages, tools=ALL_TOOL_SPECS, model=model)
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        messages.append({"role": "assistant", "content": response.content})

        # Claudeがツールを要求した場合以外は抜けます。"end_turn" ではなく "tool_use" で
        # 判定するのは、モデルが別の理由（例：max_tokens）で止まったときに、400エラーの
        # 無限ループを防ぐためです。そうしないと、空の tool_results メッセージを送り返してしまいます。
        # （サーバー側ツールを使う場合は、stop_reason == "pause_turn" も処理してください。）
        if response.stop_reason != "tool_use":
            break

        tool_calls = [block for block in response.content if isinstance(block, ToolUseBlock)]

        tool_results = []
        for tool_call in tool_calls:
            result = execute_tool(tool_call.name, tool_call.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": result,
            })

        messages.append({"role": "user", "content": tool_results})

    if eval_mode:
        return {
            "messages": messages,
            "usage": {"input_tokens": total_input_tokens, "output_tokens": total_output_tokens},
        }

    return "\n".join(block.text for block in response.content if isinstance(block, TextBlock))


print("boutiqueエージェントの準備ができました。")

### 試してみる

下のセルは、エージェントとの対話セッションを自動的に開始します。クエリを入力して、応答を確認してください。終了するには `quit` と入力します。（Run All でノートブック全体を実行する場合は、先にそのセルで `INTERACTIVE_CHAT = False` に設定するか、ここで止まったときに `quit` と入力してください。）

試してみるクエリ：
- 「ジーンズはいくらですか？」（`How much do jeans cost?`）単純なので、動くはずです
- 「Tシャツの値段は？」（`Price of a t-shirt?`）カタログのキーは `t-shirt` です。ハイフンで引っかかるでしょうか？
- 「靴はいくらですか？」（`How much for shoes?`）カタログに「shoes」はありませんが、「sneakers」はあります
- 「シャツ3枚とベルト2本で、合計はいくらですか？」（`3 shirts and 2 belts, what's my total?`）複数ツール：検索と計算
- 「ジャケットの20%オフはいくらですか？」（`What's 20% off a jacket?`）パーセントの計算が必要です
- 「何を売っていますか？」（`What do you sell?`）自分の機能を説明できるでしょうか？

In [ ]:
ANTHROPIC_ORANGE = "#E07A5F"  # ブランドのアクセントカラー。ノートブックの白地に黒の出力の中で、boutiqueエージェントが埋もれないようにする

def _display_chat_hint():
    """対話チャットの有効化方法を案内します。ノートブックではAnthropicオレンジ、スクリプト実行時は
    プレーンテキストで表示します。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックのカーネルではありません。プレーンテキストのバナーを使います")
        from IPython.display import display, HTML
        display(HTML(
            f'<div style="background:{ANTHROPIC_ORANGE};color:#fff;padding:10px 16px;'
            f'border-radius:8px;font-family:sans-serif;font-size:14px;">'
            f'🛍️ <b>The Boutique Agent</b> is ready — set '
            f'<code style="background:rgba(255,255,255,.3);padding:1px 5px;border-radius:4px;">INTERACTIVE_CHAT = True</code> '
            f'above and run this cell again to chat.</div>'
        ))
    except Exception:
        print("Boutique Agent の準備ができました。チャットするには、上で INTERACTIVE_CHAT = True に設定して、このセルをもう一度実行してください。")


INTERACTIVE_CHAT = True  # デフォルトではエージェントとチャットします。評価のセクションへ直接進むには、False にして Run All を実行してください

if not INTERACTIVE_CHAT:
    _display_chat_hint()
else:
    print("Boutique Agent の応答結果")

# 下の input() プロンプトは VS Code 自身のエディタUI（コマンドパレットのようなもの）で、
# 色は常に VS Code のテーマに従い、ここからは変更できません。手段はテキストだけなので、
# 代わりにプロンプト（prompt）の文言そのものでブランドを表現しています。
while INTERACTIVE_CHAT:
    query = input("\n🛍️  Boutiqueショッピングエージェントが稼働中です。質問を入力して Enter を押してください。エージェントを止めるには Esc を押します。")
    if not query.strip() or query.strip().lower() in ("quit", "exit", "q"):
        print("セッションを終了しました。")
        break
    print(f"\nBoutique: {run_agent(query)}")

---

## パート2：評価フレームワーク

評価は、3つの主要コンポーネントで構成されます。

```
Tasks ──> Runner ──> Graders ──> Results
```

- **タスク**は、*何を*テストするかを定義します。シナリオ、期待される動作、採点方法です。
- **ランナー**は、実行を取りまとめます。シナリオのクエリをエージェントに送り、トランスクリプトを集め、採点者を適用します。
- **採点者**は、エージェントが正しく動いたかを*チェック*し、スコアと理由を返します。

ランナーと採点者は、次の2つのセルで定義されています。両方を実行してください。コードをざっと読んで仕組みを理解してもかまいませんが、（今は）変更する必要はありません。

### 利用できる採点者

| 採点者 | チェック内容 | チェックの形式 |
|--------|---------------|---------------|
| `response_contains` | 最終テキストに文字列が含まれる（大文字小文字を区別しない） | `"49.99"` または `"jeans"` |
| `response_numeric` | 最終テキストに、許容誤差内の数値が含まれる | `{"value": 49.99, "tolerance": 0.05}` |
| `tool_use` | エージェントが特定のツールを特定の引数で呼び出した | `{"tool_name": "get_product", "arguments": {"product": "jeans"}}` |

各採点者は、2値のスコア（0 = 不合格、1 = 合格）と、その理由を返します。**すべての採点者のすべてのチェック**に通った場合にだけ、タスクは合格になります。

In [ ]:
# ── 採点者（Graders）：このセルはそのまま実行 ─────────────────────────────

import re

def grade_response_contains(result, check, context=None):
    text = result["final_text"].lower()
    target = check.lower()
    if target in text:
        return {"score": 1.0, "reason": f"Found '{check}' in response"}
    return {"score": 0.0, "reason": f"'{check}' not found in response: {result['final_text'][:200]}"}


def grade_response_numeric(result, check, context=None):
    if isinstance(check, (int, float)):
        value, tolerance = float(check), 0.01
    else:
        value = float(check["value"])
        tolerance = float(check.get("tolerance", 0.01))

    numbers = re.findall(r"-?[\d,]+\.?\d*", result["final_text"])
    for num_str in numbers:
        try:
            num = float(num_str.replace(",", ""))
            if abs(num - value) <= tolerance:
                return {"score": 1.0, "reason": f"Found {num} (expected {value} +/- {tolerance})"}
        except ValueError:
            continue
    return {"score": 0.0, "reason": f"Expected {value} (+/- {tolerance}), found: {numbers[:10]}"}


def grade_tool_use(result, check, context=None):
    tool_name = check["tool_name"]
    expected_args = check.get("arguments", None)

    for call in result["tool_calls"]:
        if call["name"] != tool_name:
            continue
        if expected_args is None:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' was called"}

        # 部分一致：指定された key だけを確認する
        actual_args = call.get("arguments", {})
        match = all(
            (isinstance(v, str) and isinstance(actual_args.get(k), str) and v.lower() == actual_args[k].lower())
            or actual_args.get(k) == v
            for k, v in expected_args.items()
        )
        if match:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' called with matching args: {expected_args}"}

    actual = [{"name": c["name"], "args": c.get("arguments", {})} for c in result["tool_calls"]]
    if expected_args:
        return {"score": 0.0, "reason": f"'{tool_name}' not called with {expected_args}. Actual: {actual}"}
    return {"score": 0.0, "reason": f"'{tool_name}' never called. Actual: {[c['name'] for c in result['tool_calls']]}"}


GRADER_REGISTRY = {
    "response_contains": grade_response_contains,
    "response_numeric": grade_response_numeric,
    "tool_use": grade_tool_use,
}

print(f"Graders loaded: {list(GRADER_REGISTRY.keys())}")

In [ ]:
# ── 評価ランナー（Eval Runner）：このセルはそのまま実行 ────────────────────

import json, os, time, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed


def parse_transcript(messages):
    """生のエージェントのトランスクリプトから final_text と tool_calls を取り出します。"""
    final_text, tool_calls = "", []
    for msg in messages:
        if msg["role"] != "assistant":
            continue
        for block in msg["content"]:
            if isinstance(block, TextBlock):
                final_text = block.text
            elif isinstance(block, ToolUseBlock):
                tool_calls.append({"name": block.name, "arguments": block.input, "id": block.id})
    # tool の結果を、対応する呼び出しに結びつける
    for msg in messages:
        if msg["role"] != "user" or not isinstance(msg["content"], list):
            continue
        for item in msg["content"]:
            if isinstance(item, dict) and item.get("type") == "tool_result":
                for call in tool_calls:
                    if call["id"] == item["tool_use_id"]:
                        call["result"] = item.get("content", "")
                        break
    return {"final_text": final_text, "tool_calls": tool_calls, "messages": messages}


def run_single_task(agent_fn, task, model=None):
    """1つのタスクを実行し、採点者を適用して、採点結果とメトリクスを含む結果を返します。"""
    start = time.time()
    try:
        raw = agent_fn(task["query"], eval_mode=True, model=model)
    except Exception:
        return {
            "task_id": task["id"], "task_description": task.get("description", ""),
            "query": task["query"], "category": task.get("category", ""),
            "error": traceback.format_exc(), "passed": False, "grades": [],
            "metrics": {"time": time.time() - start},
        }

    elapsed = time.time() - start
    result = parse_transcript(raw["messages"])
    usage = raw.get("usage", {})
    turns = sum(1 for m in raw["messages"] if m["role"] == "assistant")
    metrics = {
        "time": round(elapsed, 3), "tool_calls": len(result["tool_calls"]),
        "turns": turns, "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
    }

    grades = []
    context = {"query": task["query"], "task_id": task["id"], "model": model}
    for grader in task.get("graders", []):
        grader_fn = GRADER_REGISTRY.get(grader["type"])
        if grader_fn is None:
            grades.append({"type": grader["type"], "check": None, "score": 0.0, "reason": f"Unknown grader: {grader['type']}"})
            continue
        for check in grader.get("checks", []):
            # 例外を投げる（または不正な値を返す）grader は、そのタスクだけを失敗にします。
            # このガードがないと、run_eval の f.result() で例外が再送出され、
            # 実行全体（run）が中断してしまいます。
            try:
                grade = grader_fn(result, check, context)
                grades.append({"type": grader["type"], "check": check, "score": grade["score"], "reason": grade["reason"]})
            except Exception as exc:
                grades.append({"type": grader["type"], "check": check, "score": 0.0,
                               "reason": f"grader error: {type(exc).__name__}: {exc}"})

    passed = all(g["score"] == 1.0 for g in grades) if grades else False

    return {
        "task_id": task["id"], "task_description": task.get("description", ""),
        "query": task["query"], "category": task.get("category", ""),
        "passed": passed, "grades": grades, "metrics": metrics,
        "final_text": result["final_text"],
        "transcript": [
            block.model_dump() if hasattr(block, "model_dump") else block
            for msg in raw["messages"]
            for block in (msg["content"] if isinstance(msg["content"], list) else [msg["content"]])
        ],
    }


def run_eval(agent_fn, tasks, model=None, num_runs=1, max_workers=5):
    """評価スイート全体を実行し、構造化された結果を返します。"""
    all_runs = []
    for _ in range(num_runs):
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(run_single_task, agent_fn, t, model): t for t in tasks}
            run_results = []
            for f in as_completed(futures):
                r = f.result()
                run_results.append(r)
                mark = "PASS" if r["passed"] else ("ERROR" if r.get("error") else "FAIL")
                print(f"  [{len(run_results)}/{len(tasks)}] {r['task_id']}: {mark}", flush=True)
        task_order = {t["id"]: i for i, t in enumerate(tasks)}
        run_results.sort(key=lambda r: task_order.get(r["task_id"], 999))
        all_runs.append(run_results)
    return {"runs": all_runs, "config": {"model": model, "num_runs": num_runs, "num_tasks": len(tasks)}}


def save_results(results, directory="eval_results"):
    """評価結果をJSONファイルに保存します。"""
    os.makedirs(directory, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    model_name = results["config"].get("model") or "default"
    model_short = model_name.split("-")[1] if "-" in str(model_name) else model_name
    filename = f"{directory}/eval_{model_short}_{timestamp}.json"
    with open(filename, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Results saved to {filename}")
    return filename


def print_summary(results):
    """整形した評価結果を表示します。"""
    config = results["config"]
    print(f"{'=' * 60}")
    print(f"EVAL RESULTS: {config['num_tasks']} tasks, {config['num_runs']} run(s)")
    if config.get("model"): print(f"Model: {config['model']}")
    print(f"{'=' * 60}\n")

    for run_idx, run in enumerate(results["runs"]):
        if config["num_runs"] > 1: print(f"--- Run {run_idx + 1} ---")
        passed = sum(1 for r in run if r["passed"])
        total = len(run)
        print(f"Overall: {passed}/{total} passed ({passed/total*100:.0f}%)\n")

        # カテゴリ別（category）の内訳
        categories = {}
        for r in run:
            cat = r.get("category", "uncategorized")
            categories.setdefault(cat, {"passed": 0, "total": 0})
            categories[cat]["total"] += 1
            if r["passed"]: categories[cat]["passed"] += 1
        if len(categories) > 1:
            print("カテゴリ別：")
            for cat, c in sorted(categories.items()):
                print(f"  {cat}: {c['passed']}/{c['total']} ({c['passed']/c['total']*100:.0f}%)")
            print()

        # タスク（task）ごとの詳細
        print("Tasks:")
        for r in run:
            mark = "PASS" if r["passed"] else "FAIL"
            print(f"  [{mark}] {r['task_id']}: {r['task_description']}")
            for g in r.get("grades", []):
                print(f"    {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason'][:120]}")
            if r.get("error"): print(f"    Error: {r['error'][:200]}")

        # 集計（aggregate）メトリクス
        ok = [r for r in run if not r.get("error")]
        if ok:
            print(f"\nMetrics (avg): {sum(r['metrics']['time'] for r in ok)/len(ok):.2f}s, "
                  f"{sum(r['metrics']['tool_calls'] for r in ok)/len(ok):.1f} tool calls, "
                  f"{sum(r['metrics']['turns'] for r in ok)/len(ok):.1f} turns")
            print(f"Tokens: {sum(r['metrics']['input_tokens'] for r in ok):,} in, "
                  f"{sum(r['metrics']['output_tokens'] for r in ok):,} out")
        print()


def inspect_task(results, task_id, run_index=0):
    """特定のタスクの詳細（トランスクリプトを含む）を表示します。"""
    run = results["runs"][run_index]
    r = next((r for r in run if r["task_id"] == task_id), None)
    if r is None:
        print(f"Task '{task_id}' not found"); return

    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {r['task_id']}: {r['task_description']}")
    print(f"Query: {r['query']}")
    print(f"Response: {r.get('final_text', 'N/A')}\n")
    if r.get("error"): print(f"ERROR:\n{r['error']}"); return

    print("Grades:")
    for g in r["grades"]:
        print(f"  {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason']}")
    print(f"\nMetrics: {r['metrics']}")

    print("\nTranscript:")
    for item in r.get("transcript", []):
        if isinstance(item, dict):
            t = item.get("type", "?")
            if t == "text": print(f"  [text] {item.get('text', '')[:300]}")
            elif t == "tool_use": print(f"  [tool_use] {item.get('name', '?')}({item.get('input', {})})")
            elif t == "tool_result": print(f"  [tool_result] {str(item.get('content', ''))[:200]}")
            else: print(f"  [{t}] {str(item)[:200]}")
        else: print(f"  {str(item)[:200]}")


print("評価フレームワークの準備ができました。")

### 設計上のポイント

上の採点者とランナーのコードを読むと、いくつかの選択が目に留まります。

- **if/elif ではなくレジストリの dict。** 採点者は、素の dict である `GRADER_REGISTRY` から種類の名前で引きます。新しい採点者を追加するには、関数を1つ書いて dict に1エントリ足すだけで、分岐の連鎖には手を入れません。多くの評価フレームワークで使われているパターンです。

- **ランナーは print しない。** `run_eval()` はデータ構造を返し、`print_summary()` は別の関数です。これでランナーはテストしやすく、再利用しやすくなります。ランナーに手を入れなくても、別の表示に差し替えたり、JSONに保存したり、結果をダッシュボードに流したりできます。

- **エージェントは引数で渡す。** ランナーはエージェントを直接 import せず、`agent_fn` を引数として受け取ります。これで、モックでのテスト、別のエージェントへの差し替え、エージェントを計測用にラップすることが簡単になります。

- **トランスクリプトの解析はランナーの役割。** エージェントは生のメッセージを返し、ランナーが `final_text` や `tool_calls` などを取り出します。抽出ロジックの重複を避け、エージェントをすっきり保てます。エージェントは、どう評価されるかを知る必要がありません。

- **失敗した実行 ≠ 失敗したタスク。** エージェントが例外を投げた場合（APIエラー、タイムアウト）は*失敗した実行*で、`error` フィールドに記録され、採点はスキップされます。エージェントが間違った答えを返した場合は*失敗したタスク*で、採点者が通常どおり動き、何が悪かったかを報告します。この区別は重要です。エラーはインフラが壊れたこと、失敗はエージェントが間違った答えを出したことを意味します。

- **並行実行。** タスクは、I/Oバウンドなレスポンス待ちの定番である `ThreadPoolExecutor` で並列に実行されます。並列数を無制限にするとレート制限に当たります。エージェントが非同期の `AsyncAnthropic` クライアントを使う場合は、これを `asyncio.gather()` に置き換えます。

---

## パート3：評価タスクを定義する

### タスクのスキーマ

各タスクは、次のフィールドを持つ Python の dict です。

```python
{
    "id": "unique_task_id",             # Short, descriptive identifier
    "description": "What this tests",    # Human-readable description
    "query": "The user's question",      # What gets sent to the agent
    "category": "product_lookup",        # For grouping results
    "graders": [                          # List of grader declarations
        {
            "type": "response_contains",  # Which grader to use
            "checks": ["49.99"],          # What to check (one or more)
        },
    ],
}
```

### 手本

ジーンズの価格を調べられるかをチェックするタスクの例です。

```python
{
    "id": "price_jeans",
    "description": "Direct price lookup for jeans",
    "query": "How much do jeans cost?",
    "category": "product_lookup",
    "graders": [
        {"type": "response_contains", "checks": ["49.99"]},
        {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
    ],
}
```

このタスクは、2つの採点者を使っています。
1. `response_contains` は、最終的な答えに「49.99」が含まれているかを確認します
2. `tool_use` は、エージェントが実際に `get_product` を `product="jeans"` で呼び出したか（価格をでっち上げていないか）を確認します

### `tool_use` 採点者の脆さについて

ツール使用のチェックで確かめるのは、エージェントが答えを*ツールに基づいて出したか*（価格をでっち上げたり、暗算で済ませたりしていないか）であり、呼び出し順序を厳密に強制することではありません。エージェントが想定外でも妥当な経路で正しい答えに着いたなら、それで問題ありません。

`{"tool_name": "get_product"}`（引数なし）をチェックすると、エージェントがそのツールを使ったことだけを確認できます。`{"tool_name": "get_product", "arguments": {"product": "jeans"}}` をチェックすると、引数の値まで確認できます。これは、引数の値そのものがテスト対象のとき（例：同義語の解決）にだけ行ってください。

### あなたの番

ジーンズのタスクは参照用に用意してあります。次は、5つのクエリ例のタスクを作ってください。

1. 「Tシャツの値段は？」（`"Price of a t-shirt?"`）- エージェントはハイフンを正しく扱えるでしょうか。`get_product` にはどんな引数を渡すべきでしょうか。
2. 「靴はいくらですか？」（`"How much for shoes?"`）- 「shoes」はカタログにありません。ここでは何が起きるべきで、どう採点しますか。
3. 「シャツ3枚とベルト2本で、合計はいくらですか？」（`"3 shirts and 2 belts, what's my total?"`）- 複数の商品検索と計算が必要です。期待される合計はいくらでしょうか。
4. 「ジャケットの20%オフはいくらですか？」（`"What's 20% off a jacket?"`）- 商品検索と計算の両方が必要です。どの数値をチェックすべきでしょうか。
5. 「何を売っていますか？」（`"What do you sell?"`）- 自由回答です。手持ちの採点者で採点できるでしょうか。（ヒント：パート6のあとで、このタスクに戻ってきてください。）

それぞれについて、次を考えてみてください。
- どのカテゴリに属するか
- どの採点者が適切か。チェックは何にするか
- 期待される正しい答えは何か
- `tool_use` のチェックは必要か、それとも応答のチェックだけで十分か

### ✏️ あなたの番：評価タスクを、このセルに書く

各タスクの dict には、**id**、**query**（エージェントに送るプロンプト）、合否を決める**チェック**付きの**採点者**が必要です。手本はそのまま残し、コメントアウトされたテンプレートを新しいタスクごとにコピーして、`# ✏️ ADD YOUR TASKS BELOW` というマーカーの下に書き足してください。

In [ ]:
# ✏️ あなたの番（YOUR TURN）：評価タスクを、このリストに書きます。
# 各タスクには、id、エージェントに送るクエリ（プロンプト）、チェック付きの採点者が必要です。
tasks = [
    # ── 参照タスク（Reference task）─────────────────────────────────
    {
        "id": "price_jeans",
        "description": "ジーンズの価格を直接検索する",
        "query": "ジーンズはいくらですか？",
        "category": "product_lookup",
        "graders": [
            {"type": "response_contains", "checks": ["49.99"]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
        ],
    },

    # ── テンプレート（Template）：新しいタスクごとにこの骨組みをコピー ─────────
    # {
    #     "id": "short_unique_id",
    #     "description": "このタスクが何をテストするか",
    #     "query": "エージェントに送るユーザーメッセージそのもの",
    #     "category": "group_name_for_results",
    #     "graders": [
    #         {"type": "which_grader_to_use", "checks": ["チェックする内容"]},
    #     ],
    # },

    # ── 次のクエリのタスクを作ってください（Build tasks）────────────────────

    # 1. "Tシャツの値段は？"（Price of a t-shirt?）

    # 2. "靴はいくらですか？"（How much for shoes?）

    # 3. "シャツ3枚とベルト2本で、合計はいくらですか？"（3 shirts and 2 belts, what's my total?）

    # 4. "ジャケットの20%オフはいくらですか？"（What's 20% off a jacket?）

    # 5. "何を売っていますか？"（What do you sell?）

    # ✏️ ADD YOUR TASKS BELOW

]

---

## パート4：評価を実行する

それでは評価を実行して、エージェントの成績を見てみましょう。結果はJSONファイルとしても保存されるので、実行どうしを比較できます。

In [ ]:
results = run_eval(run_agent, tasks)
print_summary(results)
save_results(results)

### 結果を調べる

サマリーからは、*何が*合格し、何が不合格だったかがわかります。*なぜ*かを理解するには、個々のタスクを調べます。

不合格になったタスクでは、次を見てください。
- エージェントは実際に何と応答したか
- どのツール呼び出しを、どの引数で行ったか
- どこで間違えたか：ツールの選択、引数の形式、最終的な答えのどれか

合格したタスクでは、次を問いかけます。正しい理由で合格しているか。たまたま合格していないか。

不合格になったタスクでは、こう問いかけます。その判定は公平に感じられるか。

In [ ]:
# 調べたいタスクIDに置き換えてください
inspect_task(results, "price_jeans")

### ベースラインを確立する

LLMは非決定的です。最新のClaudeモデルでは temperature のようなサンプリングパラメータ自体が公開されておらず、そもそも設定を固定しても同じ出力になる保証はありませんでした。変更を加える前に、評価を複数回実行して、信頼できるベースラインを取ってください。

In [ ]:
baseline = run_eval(run_agent, tasks, num_runs=5)
print_summary(baseline)

---

## パート5：エージェントを改善する

評価結果から学んだことを使って、エージェントを改善します。

変更できるもの：
- **ツールの説明。** 現在の仕様は意図的にあいまいにしてあります。Claudeがツールを正しく使うには、どんな情報があるとよいでしょうか。
- **システムプロンプト。** `"You are a helpful assistant."` では、ショッピングアシスタントであるという文脈がClaudeに一切伝わりません。
- **エラー処理。** 商品が見つからないと何が起きるか。無効な演算子が使われたらどうなるか。
- **ツールの実装。** ツールは、入力の形式にもっと寛容にできないか。

変更したら、**上にスクロールして戻り、エージェントのセルを再実行**してから、評価を再実行し、変更が効いたかを確認します。

---

## パート6：LLM-as-Judge 採点者を追加する

クエリによっては、決定的な採点者では採点できません。次のようなものです。
- *「何を売っていますか？」* カタログの説明の仕方は、正解が何通りもあります。
- *「シャツ2枚とジャケット1着では、どちらがお得ですか？」* 答えには数値だけでなく推論が必要です。
- *「100ドルあります。何を買えばいいですか？」* 妥当なおすすめなら、どれも正解です。

こうしたクエリには、**LLM-as-Judge**、つまりClaude自身を使って応答を評価する採点者が必要です。

### 契約（インターフェース）

採点者のインターフェースは、決定的な採点者と同じです。

```python
def grade_llm_judge(result, check, context=None):
    # check: a string describing what to evaluate, e.g.
    #   "Response lists available product categories"
    #   "Response correctly identifies the better deal with reasoning"
    #
    # context: includes the original query (context["query"]) and task metadata
    #
    # Returns: {"score": 0.0 or 1.0, "reason": "..."}
```

各チェックは、自然言語で書いた基準です。採点者は、エージェントの応答と基準をClaudeに送り、2値の合否判定を求めます。

### 実装のヒント

- **プロンプトの構成。** ジャッジには、元のクエリ、エージェントの応答（`result["final_text"]`）、基準（`check`）を渡します。基準を満たしているかの評価を求めます。
- **構造化された答えを強制する。** 1行目に `PASS` か `FAIL`、続けて理由を返すよう求めます。これで解析が簡単になります。
- **1つずつにする。** 各チェックは単一の基準です。1回のLLM呼び出しで複数の基準を評価させないでください。

### 実装する

下の採点者を仕上げて登録し、それを使うタスクを書いてください。

In [ ]:
# LLM-as-Judge 採点者を実装する
#
# 構造化出力（output_config.format）を使うので、判定は enum 付きの検証済みJSONで返ります。
# 自由記述の "PASS"/"FAIL" を文字列解析する必要はありません。
# サンプリングパラメータも使いません（temperature は最新モデルで廃止されています。
# 決定性はダイヤルではなく、schema が担保します）。

JUDGE_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "verdict": {"type": "string", "enum": ["PASS", "FAIL"]},
            "reason": {"type": "string", "description": "理由を1文で。"},
        },
        "required": ["verdict", "reason"],
        "additionalProperties": False,
    },
}


def grade_llm_judge(result, check, context=None):
    # TODO：この採点者を実装する
    #
    # ステップ1（Step 1）：ジャッジ用のプロンプトを作る
    #   - 含めるもの：context["query"]、result["final_text"]、チェックの基準
    #
    # ステップ2：構造化された判定付きでClaudeを呼び出す
    #   - response = client.messages.create(model=FAST_MODEL, ...,
    #         output_config={"format": JUDGE_SCHEMA})
    #
    # ステップ3（Step 3）：判定を読み取る
    #   - data = json.loads(response.content[0].text)
    #   - 返す値：{"score": 1.0 if data["verdict"] == "PASS" else 0.0, "reason": data["reason"]}
    #
    # 未実装の judge が、実行をクラッシュさせずに読みやすく失敗するようにするための仮の実装です。
    # 自分の実装（implementation）に置き換えてください：
    return {"score": 0.0,
            "reason": "grade_llm_judge はまだ実装されていません。「LLM-as-Judge 採点者を実装する」のセルで作ってください"}


# runner が使えるように登録する
GRADER_REGISTRY["llm_judge"] = grade_llm_judge

In [ ]:
# LLM-as-Judge 採点者を使うタスクを追加する

llm_judge_tasks = [
    # {
    #     "id": "capabilities",
    #     "description": "エージェントが自分の機能を説明する",
    #     "query": "何を手伝ってもらえますか？",
    #     "category": "capabilities",
    #     "graders": [
    #         {"type": "llm_judge", "checks": [
    #             "応答が、商品の価格（price）を調べられることに触れている",
    #             "応答が、計算（calculation）を実行できることに触れている",
    #         ]},
    #     ],
    # },
]

# 両方のタスクセット（task sets）で評価を実行する
# all_tasks = tasks + llm_judge_tasks
# results = run_eval(run_agent, all_tasks)
# print_summary(results)

---

## 発展課題

早く終わった方へ。おおよそ難易度の順にアイデアを並べました。

### すぐできること
- **ネガティブなテストケース。** エージェントがうまく扱うべきクエリ：「ピアノはいくらですか？」（カタログにない）、「人生の意味は？」（話題外）。
- **タスクの検証。** 評価を実行する前に、すべてのタスクの dict に必要なフィールドがあるかをチェックする関数を書く。
- **Claudeでタスクを生成する。** カタログと、ここまでに作ったパターンをもとに、追加の評価タスクを生成するよう Claude Code に頼む。

### モデルの比較
- **Haiku vs Sonnet vs Opus。** 同じ評価を複数のモデルで実行します。より高性能なモデルは、出来の悪いツール仕様を補えるでしょうか。`run_eval(run_agent, tasks, model="claude-sonnet-5")` を使って結果を比較します。
- **コスト vs 精度。** トークン使用量と、1回の実行あたりの推定コストを集計する。Haikuがタスクの90%に合格するなら、Sonnetの追加コストに見合うでしょうか。

### より良い採点者
- **効率の採点者。** 正しさだけでなく効率も採点する：エージェントは最小限のツール呼び出し回数で済ませたか。
- **採点者の自動検出。** 手作業の `GRADER_REGISTRY` dict を、関数を自動登録する `@grader` デコレーターに置き換える。

### 分析と可視化
- **結果の可視化。** カテゴリ別の合格率や、モデル間の比較をグラフ（matplotlib、plotly）にする。
- **ベースラインの比較。** 保存した2つのJSON結果を並べて差分を取り、リグレッションにフラグを立てる。
- **Pass@k と pass^k。** 各タスクをk回実行します。Pass@k は「少なくとも1回合格」、pass^k は「毎回合格」です。不安定なタスクと、確実に壊れているタスクを見分けるのに役立ちます。
- **統計的な頑健性。** 複数回の実行にわたる分散、信頼区間、統計的有意性を計算する。

### ハーネスの改善
- **最大ターン数のガード。** ターン数の上限を加えて、エージェントが無限ループしないようにする。
- **タスクごとのタイムアウト。** 時間がかかりすぎるタスクを強制終了する。
- **バックオフ付きリトライ。** 一時的なAPIエラーをうまく処理する。
- **キャッシュ。** クエリとエージェント設定のハッシュを使って、変更のないタスクの再実行を避ける。
- **進捗の表示。** 評価の実行中に、ライブのプログレスバーを表示する。
- **パラメータスイープ。** プロンプト、エフォートレベル、モデルなどの直積を取る。

---

## このセッションを越えて

### 単純化した点

この評価の設計上の選択には、意図的に単純にしたものがあります。本番のシステムは次のように違います。

- **エージェントの計測。** 完全なトランスクリプトを返すように、エージェントに `eval_mode` フラグを追加しました。実際には、エージェントには一切手を入れたくありません。本番のフレームワークは、エージェントをラップしてAPI呼び出しを傍受し、エージェントのコードに触れずにトランスクリプトを作ります。

- **オブザーバビリティ。** ここでは、完全なトランスクリプトをフラットなリストとして記録しています。本番のシステムは、OpenTelemetry のスパンツリーを使って、タイミング、入れ子、メタデータを含む構造化されたトレースを取ります。これにより、*何が*起きたかだけでなく、各ステップに*どれだけ時間がかかり*、呼び出しがどう入れ子になっていたかまで見えます。

- **環境の分離。** ここでのエージェントは、評価と同じプロセスで動いています。コードを実行する評価（SWE-bench など）では、3層の Docker パターンが標準です。(1) OS とランタイムを含むベースイメージ、(2) 依存関係をインストールした環境イメージ（ハッシュでコンテンツアドレス化され、タスク間で共有）、(3) タスクごとの、特定のリポジトリのチェックアウトを含むインスタンスイメージ。

- **タスクの形式。** ここではインラインの Python dict を使っています。本番のハーネスは通常、JSONLファイル（1行に1タスク）、外部ファイルを参照するYAML、またはデータベースを使ったタスクストアを使います。

### 既存のフレームワーク

評価ハーネスをゼロから作る必要はありません。いくつものフレームワークが、最初から使える形で提供しています。

- **Promptfoo。** 使いやすいCLIとWeb UIを備えたオープンソースの評価フレームワーク。
- **Braintrust。** ログ、トレーシング、比較ツールを備えたホスト型の評価プラットフォーム。
- **LangSmith。** LangChainエコシステムのトレーシング・評価プラットフォーム。
- **Harbor。** LLMの安全性に焦点を当てたオープンソースの評価フレームワーク。

Anthropic社内では、**eval-toolbox** を使って、**eval-registry** に評価を追加しています。このセッションの自然な次のステップは、自分の評価をこれらのフレームワークのいずれかに移行することです。